In [1]:
import pandas as pd
import glob

# 이용데이터 로드 (파일명 타임스탬프가 바뀌어도 되게 glob으로 최신 파일 선택)
usage = pd.read_csv(sorted(glob.glob('../tassue_usage_2*.csv'))[-1])

# 문자열 → 날짜 타입 변환 후 날짜만 분리
usage['event_time'] = pd.to_datetime(usage['event_time'])
usage['date'] = usage['event_time'].dt.normalize()   # 시각 떼고 날짜만

print(f"기간: {usage['date'].min().date()} ~ {usage['date'].max().date()}")
print(f"총 {len(usage):,}행")

기간: 2025-01-01 ~ 2026-04-01
총 211,035행


In [2]:
# 날짜별로 (총 이용건수, 데이터 있는 시간대 수)를 집계
# 시간커버가 낮으면 수집이 끊겼던 날일 가능성이 있음
usage['hour'] = usage['event_time'].dt.hour
daily = usage.groupby('date').agg(
    총건수=('qty', 'sum'),
    시간커버=('hour', 'nunique'),   # 그날 기록이 있는 시간대 개수 (최대 24)
)

# 달력상 있어야 할 날짜와 비교해서 통째로 빠진 날 찾기
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
missing_days = full_range.difference(daily.index)

print(f"있어야 할 일수 {len(full_range)} / 실제 {len(daily)} / 통째로 빠진 날 {len(missing_days)}")
print("빠진 날 월별:", pd.Series(missing_days).dt.to_period('M').value_counts().sort_index().to_dict())

있어야 할 일수 456 / 실제 420 / 통째로 빠진 날 36
빠진 날 월별: {Period('2025-02', 'M'): 2, Period('2025-03', 'M'): 3, Period('2025-12', 'M'): 31}


In [4]:
# 제외 규칙:
#  - 통째로 빠진 36일은 daily에 아예 없으므로 자동 제외됨
#  - 학습 구간을 미세먼지가 있는 2026-02-28까지로 제한
#    (2026-03은 에어코리아 확정자료 공개 후 추가)
#  - 이용량이 적어도 비/명절 때문인 날은 진짜 수요이므로 유지!
END_DATE = '2026-02-28'

valid_dates = daily[daily.index <= END_DATE].index

print(f"유효 날짜: {len(valid_dates)}일")   # 388일이 나와야 정상

# 다음 단계들이 계속 쓸 수 있게 파일로 저장
pd.Series(valid_dates, name='date').to_csv('valid_dates.csv', index=False)
print("저장: valid_dates.csv")

유효 날짜: 388일
저장: valid_dates.csv


In [8]:
print(usage.head(10).to_string())

   id event_type          event_time station_id  qty       date  hour
0   1         반납 2025-01-01 05:10:00     ST0685    2 2025-01-01     5
1   2         대여 2025-01-01 05:15:00     ST0639    2 2025-01-01     5
2   3         반납 2025-01-01 05:25:00     ST0065    2 2025-01-01     5
3   4         반납 2025-01-01 05:25:00     ST0627    2 2025-01-01     5
4   5         대여 2025-01-01 05:35:00     ST1302    2 2025-01-01     5
5   6         대여 2025-01-01 05:55:00     ST0627    2 2025-01-01     5
6   7         대여 2025-01-01 07:05:00     ST0685    2 2025-01-01     7
7   8         반납 2025-01-01 07:25:00     ST0685    2 2025-01-01     7
8   9         대여 2025-01-01 07:40:00     ST0720    2 2025-01-01     7
9  10         반납 2025-01-01 07:50:00     ST0065    2 2025-01-01     7


In [9]:
# 1단계에서 확정한 388일만 남긴다 (2026-03 이후, 수집누락일 제거)
valid_dates = pd.to_datetime(pd.read_csv('valid_dates.csv')['date'])
usage_v = usage[usage['date'].isin(valid_dates)].copy()

print(f"필터 전 {len(usage):,}행 → 필터 후 {len(usage_v):,}행")

필터 전 211,035행 → 필터 후 193,099행


In [10]:
# groupby: 같은 (스테이션, 날짜, 시간, 이벤트종류)끼리 qty 합산
# ※ 행 개수(count)가 아니라 qty 합(sum)! (5분 묶음 안에 여러 건이 있으므로)
agg = usage_v.groupby(['station_id', 'date', 'hour', 'event_type'])['qty'].sum()

# pivot: 세로로 쌓인 대여/반납을 가로 두 컬럼으로 펼치기
target = agg.unstack('event_type')          # 대여, 반납이 컬럼이 됨
target.columns = ['rent_cnt', 'return_cnt'] # 컬럼명 영문으로 (대여→rent, 반납→return)

# 그 시간에 대여만 있고 반납은 없었으면 반납이 NaN이 됨 → 실제로는 0건이므로 0으로
target = target.fillna(0).astype(int).reset_index()

print(f"집계 결과: {len(target):,}행")
print(target.head(10).to_string())

집계 결과: 68,619행
  station_id       date  hour  rent_cnt  return_cnt
0     ST0056 2025-01-01    16         0           2
1     ST0056 2025-01-01    17         4           4
2     ST0056 2025-01-01    18         2           2
3     ST0056 2025-01-01    19         2           0
4     ST0056 2025-01-02     9         0           2
5     ST0056 2025-01-02    10         2           0
6     ST0056 2025-01-02    13         2           2
7     ST0056 2025-01-02    16         2           2
8     ST0056 2025-01-02    18         0           2
9     ST0056 2025-01-02    19         4           2


In [11]:
# 원본과 대조: 2025-01-01 5시 ST0685는 반납 2+대여... 직접 눈으로 확인
print("원본 이벤트:")
print(usage_v[(usage_v['station_id']=='ST0685') & (usage_v['date']=='2025-01-01') & (usage_v['hour']==5)].to_string())

print("\n집계 결과:")
print(target[(target['station_id']=='ST0685') & (target['date']=='2025-01-01') & (target['hour']==5)].to_string())

원본 이벤트:
   id event_type          event_time station_id  qty       date  hour
0   1         반납 2025-01-01 05:10:00     ST0685    2 2025-01-01     5

집계 결과:
      station_id       date  hour  rent_cnt  return_cnt
38480     ST0685 2025-01-01     5         0           2


In [12]:
import itertools

# 388일 × 24시간 × 19스테이션의 모든 조합 생성
stations = usage_v['station_id'].unique()

grid = pd.DataFrame(
    itertools.product(stations, valid_dates, range(24)),
    columns=['station_id', 'date', 'hour']
)

print(f"격자: {len(grid):,}행")   # 19 × 388 × 24 = 176,928이 나와야 함

격자: 176,928행


In [13]:
# left join: 격자를 기준으로 붙이므로, 집계에 없던 조합은 NaN이 됨
base = grid.merge(target, on=['station_id', 'date', 'hour'], how='left')

# NaN = 그 시간에 아무 기록 없음 = 수요 0
base['rent_cnt'] = base['rent_cnt'].fillna(0).astype(int)
base['return_cnt'] = base['return_cnt'].fillna(0).astype(int)

print(f"최종: {len(base):,}행")
print(f"수요 0인 행 비율: {(base['rent_cnt'].eq(0) & base['return_cnt'].eq(0)).mean()*100:.0f}%")
print(base.head(10).to_string())

최종: 176,928행
수요 0인 행 비율: 61%
  station_id       date  hour  rent_cnt  return_cnt
0     ST0685 2025-01-01     0         0           0
1     ST0685 2025-01-01     1         0           0
2     ST0685 2025-01-01     2         0           0
3     ST0685 2025-01-01     3         0           0
4     ST0685 2025-01-01     4         0           0
5     ST0685 2025-01-01     5         0           2
6     ST0685 2025-01-01     6         0           0
7     ST0685 2025-01-01     7         2           2
8     ST0685 2025-01-01     8         0           4
9     ST0685 2025-01-01     9         0           0


In [14]:
# 검증 1: 총 건수가 집계 전과 같아야 함 (0을 채웠을 뿐 건수가 변하면 안 됨)
print("집계본 대여 합:", target['rent_cnt'].sum(), "/ 격자본 대여 합:", base['rent_cnt'].sum())
print("집계본 반납 합:", target['return_cnt'].sum(), "/ 격자본 반납 합:", base['return_cnt'].sum())

# 검증 2: 중복 조합이 없어야 함
print("중복:", base.duplicated(subset=['station_id','date','hour']).sum())

# 타슈 쪽 전처리 완료본 저장
base.to_csv('target_base.csv', index=False)
print("저장: target_base.csv")

집계본 대여 합: 120628 / 격자본 대여 합: 120628
집계본 반납 합: 121030 / 격자본 반납 합: 121030
중복: 0
저장: target_base.csv
